## Download the data
#### Source: _Yahoo Finance_

In [ ]:
from dataclasses import dataclass
from datetime import datetime
import pandas as pd
from typing import TypedDict
import yfinance as yf

@dataclass
class TickerHistory(TypedDict):
    name: str
    data: pd.DataFrame

def download_tickers_history(start_date: datetime, end_date: datetime, tickers: list[str]):   
    df_list = yf.download(
        tickers=tickers,
        start=start_date,
        end=end_date,
        interval="1d",
        group_by="ticker",
        auto_adjust=True,
        threads=True,
        progress=True
    );

    return df_list;

# set the date range for the historic data
start_date = datetime(year=2020, month=1, day=1)
end_date = datetime(year=2025, month=12, day=31)
history = download_tickers_history(start_date, end_date, ['NVDA']);

history.NVDA

## Basic data analytics

#### Returns
Working with raw prices (Close) is impossible in statistics, as they are non-stationary (the trend may be up or down, and the mathematical expectation may change over time). The first thing to do is to calculate the logarithmic return.

$$R_t = \ln(P_t / P_{t-1}) = \ln(P_t) - \ln(P_{t-1}),$$

where $t$ - time (day)

In [ ]:
import numpy as np;
import matplotlib.pyplot as plt;
from scipy.stats import norm

nvda = history.NVDA;
prices = nvda['Close'].to_numpy()
log_returns = np.diff(np.log(prices))

mu = np.mean(log_returns)
sigma = np.std(log_returns)

# data visualization
plt.figure(figsize=(10, 6))

plt.hist(
    log_returns, 
    density=True,
    bins=100,
    linewidth=0.5,
    edgecolor='w',
    label='Log Returns'
);

x = np.linspace(mu - 4*sigma, mu + 4*sigma, 500)
gauss = norm.pdf(x, loc=mu, scale=sigma)

plt.plot(
    x,
    gauss, 
    color='#d62728', 
    linewidth=2.5, 
    linestyle='--', 
    label=f'Gauss distribution\n($\mu={mu:.4f}$, $\sigma={sigma:.4f}$)'
)

plt.xlabel('Logarithmic Returns ($R_t$)', fontsize=12)
plt.ylabel('Distribution Density', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()

plt.show();


### Find the distribution

We're gonna use some quantitative criteria (Skewness and Excess Kurtosis), and also test test the hypothesis that the observed sample of returns follows a normal distribution using two different criterias:
* Pirson (Сhi-square)
* Shapiro-Wilk test

In [ ]:
confidence_lvl = 0.95
alpha = 1 - confidence_lvl

In [ ]:
from scipy.stats import skew, kurtosis

skewness = skew(log_returns)
excess_kurtosis = kurtosis(log_returns)

print("Skewness:", skewness)
print("Excess Kurtosis:", excess_kurtosis)

if skewness in [0.1, 0.5] or excess_kurtosis > 0:
    print("The distribution is likely NOT Normal.")
else:
    print("Looks like it is Normal distribution.")

In [ ]:
from scipy.stats import chisquare

# Normal with mu=0.0023 and sigma=0.0335
n = len(log_returns)

# split into intervals (bins)
num_bins = 20
observed_freq, bin_edges = np.histogram(log_returns, bins=num_bins)

# generate random 
cdf_values = norm.cdf(bin_edges, loc=mu, scale=sigma) # Gauss CDF
expected_prob = np.diff(cdf_values) # probability of falling into the i-th interval
expected_freq = expected_prob * n

expected_freq = expected_freq * (observed_freq.sum() / expected_freq.sum())

chi2_stat, p_value = chisquare(f_obs=observed_freq, f_exp=expected_freq, ddof=2)

print(f"Criterion statistics: {chi2_stat:.3f}")
print(f"P-value: {p_value:.5e}")

if p_value < alpha:
    print("Hypothesis H0 is rejected: The distribution IS NOT Normal.")
else:
    print("Hypothesis H0 IS NOT rejected: Looks like it is Normal distribution.")


In [ ]:
from scipy.stats import shapiro

stat, p_value = shapiro(log_returns)
print(f"p-value: {p_value}")

if p_value < alpha:
    print("Hypothesis H0 is rejected: The distribution IS NOT Normal.")
else:
    print("Hypothesis H0 IS NOT rejected: Looks like it is Normal distribution.")



### Prices log-normal distribution

In [ ]:
from scipy.stats import lognorm

log_prices = np.log(prices)
mu = np.mean(log_prices)
sigma = np.std(log_prices)
scale = np.exp(mu)

# data visualization
plt.figure(figsize=(10, 6))

plt.hist(
    prices, 
    density=True,
    bins=100,
    linewidth=0.5,
    edgecolor='w',
    label='Prices'
);

x = np.linspace(np.min(prices), np.max(prices), 500)
log_norm = lognorm.pdf(x, s=sigma, scale=scale)

plt.plot(
    x,
    log_norm, 
    color="#6dd627", 
    linewidth=2.5, 
    linestyle='--', 
    label=f'Log-normal distribution\n($\mu={mu:.4f}$, $\sigma={sigma:.4f}$)'
)

plt.xlabel('Close Prices ($P_t$)', fontsize=12)
plt.ylabel('Distribution Density', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()

plt.show();


In [ ]:
from scipy.stats import chisquare

# Log-normal with mu=0.3.5158 and sigma=1.0.171
n = len(prices)
log_prices = np.log(prices)
mu = np.mean(log_prices)
sigma = np.std(log_prices)
scale = np.exp(mu)

# split into intervals (bins)
num_bins = 20
observed_freq, bin_edges = np.histogram(prices, bins=num_bins)

cdf_values = lognorm.cdf(bin_edges, s=sigma, scale=scale) # log-normal CDF
expected_prob = np.diff(cdf_values) # probability of falling into the i-th interval
expected_freq = expected_prob * n

expected_freq = expected_freq * (observed_freq.sum() / expected_freq.sum())

chi2_stat, p_value = chisquare(f_obs=observed_freq, f_exp=expected_freq, ddof=2)

print(f"Criterion statistics: {chi2_stat:.3f}")
print(f"P-value: {p_value:.5e}")

if p_value < alpha:
    print("Hypothesis H0 is rejected: The distribution IS NOT Log-Normal.")
else:
    print("Hypothesis H0 IS NOT rejected: Looks like it is Log-Normal distribution.")

## Volatility and volatility clustering
I'm going to think about volatility as the _std_ over the _returns_

Trying to build a Rolling Volatility over a window of, 20 trading days (roughly equivalent to a trading month)

In [ ]:
import numpy as np;
import matplotlib.pyplot as plt;

nvda = history.NVDA
prices = nvda['Close']
log_returns = np.diff(np.log(prices))

window = 20

rolling_vol_daily = np.array([np.std(log_returns[i:i+window]) for i in range(len(log_returns) - window)]) # len(log_returns) - window), since log_returns[i:i+window]

TRADING_DAYS_PER_YEAR = 252
rolling_vol_annual = rolling_vol_daily * np.sqrt(TRADING_DAYS_PER_YEAR) * 100 # (%)

dates = prices.index[window + 1:]

# data visualization
plt.figure(figsize=(10, 6))

plt.plot(
    dates,
    rolling_vol_annual,
    linewidth=1,
    label='20-Day Rolling Volatility (Annualized)'
);

plt.title('NVDA: 20-Day Rolling Volatility')
plt.xlabel('Periods (1 period = 1 day)', fontsize=12)
plt.ylabel('Annualized Volatility (%)', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()

plt.show()

## Volatility x Volume

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

nvda = history.NVDA

log_returns = np.diff(np.log(nvda.Close.values))
abs_log_returns_perc = np.abs(log_returns) * 100 # (%)

volume = nvda.Volume.values[1:]

v_min, v_max = np.min(volume), np.max(volume)
size = 15 + (volume - v_min) / (v_max - v_min) * 185

# data visualization
fig, ax = plt.subplots(figsize=(10, 6));

scatter = ax.scatter(volume, abs_log_returns_perc, c=abs_log_returns_perc, s=size, cmap="Spectral_r")

# disable scientific notation & offset on X-axis to show full numbers
ax.ticklabel_format(style='plain', useOffset=False, axis='x')

ax.set_title("NVDA: Volume vs. Volatility (MDH Funnel Effect)", fontsize=14, fontweight='bold')
ax.set_xlabel("Daily Volume (in millions)")
ax.set_ylabel("Absolute Log Returns (%)")

ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f'{x/1e6:.0f}M'))
legend1 = ax.legend(*scatter.legend_elements(num=5),
                    loc="upper left", title="Abs Return (%)")
ax.add_artist(legend1)

handles, labels = scatter.legend_elements(prop="sizes", alpha=0.6, num=5)
legend2 = ax.legend(handles, labels, loc="upper right", title="Volume")

ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

plt.show()


## Intraday Features
Global models often do not support what happens within a single trading day.

So we can visualize and analyze the gaps, and fluctuations that are happening within a day.

### High-to-Low Spread
$$\text{TR}_t = \max\left( \text{High}_t - \text{Low}_t,\; \vert{}\text{High}_t - \text{Close}_{t-1}\vert{},\; \vert{}\text{Low}_t - \text{Close}_{t-1}\vert{} \right)$$
$$Spread_t = [(High_t - Low_t) / Close_t] * 100\%$$

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

nvda = history.NVDA

low = nvda.Low
high = nvda.High
close = nvda.Close

spread_pct = ((high - low) / close) * 100 # (%)
atr_14 = spread_pct.rolling(window=14).mean()
dates = nvda.index

# data visualization
fig, ax = plt.subplots(figsize=(12, 6))

ax.bar(
    dates,
    spread_pct,
    color='lightgreen',
    alpha=1,
    linewidth=1,
    label='Daily spread (%)'
);

plt.plot(
    dates,
    atr_14,
    color='darkred',
    alpha=0.7,
    linewidth=2,
    label='14-Day Average Range (ATR Trend)'
);

plt.title('NVDA: Daily price change (%)')
plt.xlabel('Periods (1 period = 1 day)', fontsize=12)
plt.ylabel('Daily Change (% of close prices)', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()

plt.tight_layout()
plt.show()


### Open-to-Close Gap
Shows how much risk the market carries overnight while the exchange is closed.
$$ln(Open_t / Close_{t-1})$$

In [ ]:
nvda = history.NVDA

open = nvda.Open
close = nvda.Close

overnight_gap = np.log(open / close.shift(1)) * 100
overnight_gap = overnight_gap
dates = nvda.index

# data visualization
fig, ax = plt.subplots(figsize=(12, 6))

ax.bar(
    dates,
    overnight_gap,
    color='darkblue',
    alpha=0.7,
    linewidth=2.5,
    label='Nightly price shift (%)'
);

plt.title('NVDA: Overnight gap (%)')
plt.xlabel('Periods (1 period = 1 day)', fontsize=12)
plt.ylabel('Overnight Change (% from Previous Close)', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()

plt.show()

If Skewness $> 0$, positive gaps (upwards) occur more frequently or are more aggressive.

A Kurtosis value $> 3$ will immediately show heavy-tail risk, the probability of waking up with a $-10\%$ gap.

In [ ]:
gap_skew = overnight_gap.skew()
gap_kurt = overnight_gap.kurtosis()

fig, ax = plt.subplots(figsize=(12, 6))

plt.hist(
    overnight_gap, 
    density=True,
    bins=200,
    linewidth=1,
    color='green',
    edgecolor='w',
    label='Open-Close Gap'
)

plt.title(f'NVDA: Overnight gap distr (Skew: {gap_skew:.2f}, Kurt: {gap_kurt:.2f})')
plt.xlabel('Overnight gap (%)', fontsize=12)
plt.ylabel('Distribution Density', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)

plt.show()

#### Overnight Volatility

In [ ]:
overnight_vol_20 = overnight_gap.rolling(window=20).std() # 20-day window of overnight gap volatility
dates = nvda.index

fig, ax = plt.subplots(figsize=(12, 6))

plt.plot(
    dates,
    overnight_gap,
    linewidth=1,
    color='purple',
    label='20-Day rolling overnight volatility (std)'
)

plt.title(f'NVDA: Overnight Volatility')
plt.xlabel('Period', fontsize=12)
plt.ylabel('Std Dev (%)', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)

plt.show()


#### Gap Continuation / Mean Reversion
The main metric for *swing trading* is what happens to the price *AFTER* a gap during the trading day:

* **Fade**: If the price opened with an upward gap, but then fell back to the previous day's close.
* **Continuation**: If after an upward gap, the price continues to rise until the close.

##### Strategies
* If the correlation between gap and the last day's return is **negative** &mdash; the major strategy on the market is the _"filling the gaps"_ (sell the gap up)
* If the correlation between gap and the last day's return is **positive** &mdash; the major _"impulsive strategy"_ is the dominant on the market (buy the gap up, since the price continues to grow)

In [ ]:
import seaborn as sns
from colorama import init, Style

init(autoreset=True)

intraday_return = np.log(close / open) * 100 # (%)
dates = nvda.index.year

gap_data = pd.DataFrame({
    'Overnight_Gap': overnight_gap,
    'Intraday_Return': intraday_return,
}).dropna()

# Pearson correlation
p_corr = gap_data['Overnight_Gap'].corr(gap_data['Intraday_Return'])
print(f"Pearson correlation (Linear): {Style.BRIGHT} {p_corr:.3f}")

# Trim the outliers
p_low = gap_data['Overnight_Gap'].quantile(0.01)
p_high = gap_data['Overnight_Gap'].quantile(0.99)

filtered_data = gap_data[
    (gap_data['Overnight_Gap'] > p_low) & 
    (gap_data['Overnight_Gap'] < p_high)
]

print(f"Pearson correlation (no outliers): {Style.BRIGHT} {filtered_data['Overnight_Gap'].corr(filtered_data['Intraday_Return']):.3f}")

# Spearman correlation
p_corr = gap_data['Overnight_Gap'].corr(gap_data['Intraday_Return'], method='spearman')
print(f"Spearman correlation (Rank): {Style.BRIGHT} {p_corr:.3f}")

# data visualization
fig, ax = plt.subplots(figsize=(11, 7))

scatter = plt.scatter(
    overnight_gap,
    intraday_return,
    c=dates,
    cmap="viridis",
    s=35
)

# zero lines for Quadrants
ax.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.7)
ax.axvline(0, color='black', linestyle='--', linewidth=1, alpha=0.7)

plt.title(f'NVDA: Overnight Gap vs. Intraday Return (Fade / Continuation)')

# linear trendline
sns.regplot(
    x='Overnight_Gap',
    y='Intraday_Return',
    data=gap_data,
    ax=ax,
    scatter=False,
    color='red',
    line_kws={'linewidth': 1.8, 'label': f'OLS Trend (r = {p_corr:.3f})'}
)

legend1 = ax.legend(*scatter.legend_elements(num=6),
                    loc="lower right", title="Gap shift (%)")
ax.add_artist(legend1)

# colorbar for Years
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Year', fontsize=11)

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()

plt.show()


## Overnight vs. Intraday
$$\text{Ratio} = \frac{\text{Var}(\text{Overnight Returns})}{\text{Var}(\text{Intraday Returns})}$$

In a majority of a public corporations a portion of the total annual return comes from the overnight gaps (in earnings and announcements), not from intraday trading.
If the variance ratio is high, holding an overdrive carries more risk.

In [ ]:
ratio = gap_data['Overnight_Gap'].var() / gap_data['Intraday_Return'].var()
print(f"Overnight vs Intraday Ratio: {ratio:.3f}")


### Close Location
Shows where the day closed relative to its low and high:
$$(Close_t - Low_t) / (High_t - Low_t)$$
A value close to 1 means that bulls (buyers) dominated at the end of the day.

In [ ]:
num_of_years = len(nvda.index) // 365
last_year_data = nvda[(num_of_years - 1) * 365:]

close_loc = (last_year_data['Close'] - last_year_data['Low']) / (last_year_data['High'] - last_year_data['Low'])
dates = last_year_data.index

# data visualization
fig, ax = plt.subplots(figsize=(12, 6))

ax.stem(
    dates,
    close_loc,
    linefmt='y-',
    markerfmt='go',
    label='Daily close localization'
);

plt.title('NVDA: Overnight Location')
plt.xlabel('Periods', fontsize=12)
plt.ylabel('Abs Overnight Location', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()

plt.show()


## Autocorrelation and Market "Memory"
_Hypothesis_: Can tomorrow’s return be predicted by today’s return?

Will just use autocorrelation function with 1, 2, 3, 5, 10, 20 days shift

* Raw Log Returns ($r_t$): Measures directional predictability (found: $≈0$).
* Squared Log Returns ($r_t^2$): Measures variance memory (found: $+0.13$).
* Absolute Log Returns ($|r_t|$): Measures magnitude memory without outlier distortion


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf

prices = nvda['Close'].to_numpy()
log_returns = np.diff(np.log(prices))

clean_data = pd.DataFrame({
    'Log_Return': log_returns,
    'Squared_Return': log_returns ** 2,
    'Abs_Return': np.abs(log_returns)
}).dropna()

lags = [1, 2, 3, 5, 10, 20]
autocorr_results = []

for lag in lags:
    autocorr_results.append({
        'Lag': lag,
        'Raw Returns (Direction)': clean_data['Log_Return'].autocorr(lag),
        'Squared Returns (r^2)': clean_data['Squared_Return'].autocorr(lag),
        'Absolute Returns (|r|)': clean_data['Abs_Return'].autocorr(lag)
    })
results_df = pd.DataFrame(autocorr_results)
print(results_df)

print('--------------- \n')

# data visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)

plot_acf(
    clean_data['Log_Return'],
    lags=30,
    ax=axes[0],
    title='Raw Returns r_t (No Memory)',
    color='r',
    alpha=0.05
)

plot_acf(
    clean_data['Squared_Return'],
    lags=30,
    ax=axes[1],
    title='Squared Returns r_t^2 (Volatility)',
    color='b',
    alpha=0.05
)

plot_acf(
    clean_data['Abs_Return'],
    lags=30,
    ax=axes[2],
    title='Absolute Returns |r_t| (Long Memory)',
    color='g',
    alpha=0.05
)

for ax in axes:
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.set_xlabel('Lag (Days)')

axes[0].set_ylabel('Autocorrelation Coefficient')
plt.tight_layout()

plt.show()


We can see larger numbers for squares and absolute returns. This demonstrates the ability of the market to have a "memory".
Despite the fact that the values differs not drastically, e.g. $|r_t - r_t^2| ≈ 0.04-0.07$ and $|r_t - |r_t|| ≈ 0.13-0.16$, this shows a statistically big difference.

 For $N ≈ 1,650$ trading days, the 95% noise boundary around zero is:
 $$±2/√(N) = ±2 / √(1650) ≈ ±0.049$$
Thus, a value of $+0.16$ is more than $3.2$ standard deviations away from zero ($p < 0.001$).

 Why $|r_t|$ is higher and decays slower than r²ₜ (The Taylor Effect):
* Squaring returns ($r_t^2$) amplifies extreme outlier days (e.g., a $15%$ earnings gap squared is $0.0225$, which is $225 ×$ larger than a $1%$ day). These outliers distort Pearson correlation.
* Taking absolute returns (|r_t|) is robust to extreme spikes. empirical finance shows that $|r_t|$ consistently exhibits higher autocorrelation and longer memory than $r_t^2$ₜ.